# 3 Build a custom dataloader

In `02_datagenerators.ipynb` you built a generator that yields infinite shuffled batches from a
tensor of random numbers. This notebook is the sequel: **the same pattern, on real data that does
not fit in memory** — and then the library version of it, which is what you will actually use.

The generators (the `yield` keyword, the nested index) are explained in `02_datagenerators.ipynb`.
If that pattern is not fresh, go back to it first; everything here builds on it.

## 3.1 The problem: images get big fast

The problem with images is that the size grows pretty fast.

In [ ]:
import numpy as np

image_size = (180, 180, 3)

for i in [1, 10, 100]:
    size = (i,) + image_size
    X = np.zeros(size)
    size_byte = X.nbytes
    print(f"Size for {i} images: {size_byte / (2**20)} MB")

Imagine what would happen if you actually have a million images! And no, the answer to this is not
"just get more RAM in the cloud".

You don't need all of it in memory at the same time. Only the batch you are training on right now
has to exist. Everything else can stay on disk until you ask for it. That is the whole idea behind
a dataloader, and it is what we will build.

## 3.2 First, an honest look at what the tooling already does

Before hand-rolling anything, let's see how bad the problem really is. We will use the Oxford
Flowers dataset from the huggingface hub — 7169 training photos of flowers in 102 classes. It is
the dataset we come back to in lesson 4 for transfer learning.

`load_dataset` downloads it once (~330 MB) and caches it; running this cell a second time is
instant.

In [ ]:
from datasets import load_dataset

flowers = load_dataset("nkirschi/oxford-flowers")
flowers

In [ ]:
train = flowers["train"]
len(train)

Now do the arithmetic from §3.1 on this dataset. Every photo is roughly 500x523 pixels, 3 colour
channels, one byte per channel.

In [ ]:
img = np.asarray(train[0]["image"])
print("one decoded image:", img.shape, f"{img.nbytes / 2**20:.2f} MB")
print(f"all {len(train)} decoded: {img.nbytes * len(train) / 2**30:.1f} GB")

About 6.5 GB of pixels. That does not fit comfortably anywhere, and yet `len(train)` answered
instantly and `train[0]` handed over an image without complaining.

So where is it? Two things are going on, and both are worth understanding:

In [ ]:
# 1. the file on disk holds *compressed* jpeg bytes, and it is memory-mapped, not read
print(f"arrow table: {train.data.nbytes / 2**20:.0f} MB")

# 2. the image column is decoded lazily -- per row, when you ask for it
print(train.features["image"])

The dataset is an [Apache Arrow](https://arrow.apache.org/) table that the OS **memory-maps**: the
file stays on disk, and pages are pulled in on demand (and can be evicted again under memory
pressure). And the `Image(decode=True)` feature means the jpeg bytes only become an array of
pixels at the moment you index that row.

So the 6.5 GB never exists. You hold 290 MB of mapped, compressed bytes, and one decoded image at
a time.

**This is the modern answer to the question this notebook opens with**, and it is a stronger
version of the original one: you don't avoid loading everything into memory by being clever with
generators, you avoid it because the storage format was designed for it. Worth knowing, because it
changes what you have to build yourself: not the storage, only the batching.

## 3.3 Building the iterator by hand

What the tooling does *not* give you for free is the training-loop behaviour: shuffled, batched,
endless. That is the pattern from `02_datagenerators`, and it is still worth writing once.

Recall the three moving parts:

- an **index list**, shuffled with `torch.randperm`, so we shuffle indices instead of data;
- a **pointer** that walks the index list and wraps around at the end, reshuffling as it goes;
- `yield`, so the whole thing is a generator and holds one batch at a time.

The only difference with lesson 2 is that `data[index]` now reaches into a memory-mapped file and
decodes a jpeg, instead of slicing a tensor that was already in RAM.

In [ ]:
import torch


def process(item) -> tuple[torch.Tensor, int]:
    """one row of the dataset -> one (image, label) pair, channel-first and scaled"""
    img = np.asarray(item["image"].resize((150, 150)))
    X = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
    return X, item["label"]


X, y = process(train[0])
X.shape, y

Note the `permute(2, 0, 1)`: the image arrives as `(H, W, C)` and pytorch wants `(C, H, W)`. That
is the channel-first convention from `01_tensors.ipynb` §1.6.

In [ ]:
def generator(dataset, batchsize: int):
    size = len(dataset)
    index_list = torch.randperm(size)
    index = 0

    while True:
        batch = []
        for _ in range(batchsize):
            if index >= size:
                # walked off the end: reshuffle and start over
                index = 0
                index_list = torch.randperm(size)
            batch.append(process(dataset[int(index_list[index])]))
            index += 1
        yield batch

In [ ]:
gen = generator(train, batchsize=32)
batch = next(gen)
len(batch), type(batch[0]), batch[0][0].shape

Same shape as lesson 2: an endless stream of shuffled batches, and the dataset can be far bigger
than memory because we only ever hold `batchsize` decoded images.

But look at what a batch actually *is* right now: a list of 32 `(X, y)` tuples. That is not what a
model wants.

## 3.4 Collating: from a list of pairs to a pair of batches

We want to untangle `[(X, y), (X, y), ...]` into `(X, X, ...), (y, y, ...)`. Think of it as
unzipping a zipper — and weirdly enough, in python we use the same command for this as we would
use to create the pairs.

In [ ]:
def batch_processor(batch):
    X, Y = zip(*batch)
    return torch.stack(X), torch.tensor(Y)

In [ ]:
X, y = batch_processor(batch)
X.shape, y.shape

A batch of 32 images, each `(3, 150, 150)`, and a vector of 32 labels.

**This function has a name in every library there is: a `collate_fn`.** It is the piece that turns
"a list of individual items" into "a batch". Remember it — in the rest of the course you will pass
one to a `DataLoader` and to `vectormesh`'s `Collate`, and it is exactly this function.

On the tensor ladder ([tensor-ladder.md](tensor-ladder.md)), collating
is the step that adds the batch axis: it takes items and gives you a batch.

## 3.5 The library version: `DataLoader`

Everything above is what `torch.utils.data.DataLoader` does — plus worker processes, so the
decoding of the next batch happens while the GPU is busy with this one.

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(items):
    return batch_processor([process(item) for item in items])


loader = DataLoader(
    train,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
)

X, y = next(iter(loader))
X.shape, y.shape

The three pieces map one to one onto what you just wrote:

| yours | `DataLoader` |
|---|---|
| `torch.randperm` + wrapping pointer | `shuffle=True` |
| `batchsize` in the inner loop | `batch_size=32` |
| `batch_processor` | `collate_fn=...` |

Two differences worth knowing:

- **A `DataLoader` is finite.** Iterating it walks the dataset once — one epoch — and stops. Your
  generator never stops. Both are used; just know which one you have.
- **`num_workers`** spawns subprocesses that prepare batches in parallel. Tempting, but on a
  laptop the workers compete with training for the same cores, and the gain here is a fraction of
  a millisecond per batch. We use `num_workers=0` in notebooks (see
  [planning/conventions.md](../../planning/conventions.md)).

## 3.6 When the data really is unbounded: streaming

Memory-mapping solves "does not fit in RAM". It does not solve "does not fit on **disk**", and it
does not solve "I want to start training before the download finishes".

For that there is `streaming=True`, which downloads nothing up front and pulls rows over the
network as you iterate. Here is FineWeb, a web-crawl text dataset. The `sample-10BT` config alone
is about 28 GB, and the full dataset is measured in terabytes.

In [ ]:
fineweb = load_dataset(
    "HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True
)
fineweb

In [ ]:
row = next(iter(fineweb))
row["text"][:300]

We just read a row out of a 28 GB dataset without downloading it. Note the type: `IterableDataset`,
not `Dataset`. That is the trade-off, and it is a real one:

| | `Dataset` (memory-mapped) | `IterableDataset` (streaming) |
|---|---|---|
| `len(ds)` | yes | **no** — the length is not known |
| `ds[5000]` | yes, instant | **no** — you can only go forwards |
| shuffling | exact, over the whole set | approximate, over a buffer |
| disk needed | the whole dataset | nothing |
| starts | after the download | immediately |

So streaming is not simply better. You give up random access and exact shuffling, which is why
the default in this course is a downloaded, memory-mapped `Dataset`, and streaming comes out only
when the data genuinely does not fit or the download is not worth waiting for.

Both feed a `DataLoader` the same way — that is the point of the abstraction.

## 3.7 What to take away

1. **The batch is the only thing that has to be in memory.** Everything else stays on disk. That
   is true whether the storage is a memory-mapped Arrow file or a stream over the network.
2. **You can build the shuffling/batching loop yourself**, and now you have — it is an index, a
   pointer and a `yield`. Knowing that is what lets you adapt when a dataset does not fit the mould.
3. **A `collate_fn` turns items into a batch.** You will write several of these in this course.
4. **`Dataset` vs `IterableDataset` is a trade-off**, not a ranking: random access and exact
   shuffling, against not needing the data locally.

> **Exercise.** Our `generator` reshuffles when the pointer wraps, so it never runs out. A
> `DataLoader` stops after one epoch and gets re-created by the training loop. Take a training run
> of 5 epochs over 7169 images with `batch_size=32`: how many times does each image get shown to
> the model in each case? Where could the two disagree, and does it matter?